# Milestone 2 — Baseline Reproduction: RAG Robustness Against Spurious Features
### Course: 4th-Year Undergraduate NLP Research Project (Research Track)
**Anchor Paper:** *Quantifying and Improving the Robustness of Retrieval-Augmented LMs Against Spurious Features in Grounding Data* (ACL 2026)  
**Author:** Siddhartha Shukla  
**GitHub Repository:** [https://github.com/SiddharthaShukla8/NLP_Project](https://github.com/SiddharthaShukla8/NLP_Project)  
**Environment:** Google Colab / Kaggle Free GPU (T4 / P100)

## Section 1: Environment Setup & Package Installation
We install official pinned versions of HuggingFace `transformers`, `datasets`, `evaluate`, and `accelerate`.

In [ ]:
!pip install -q transformers==4.35.2 datasets==2.14.6 evaluate==0.4.1 accelerate==0.24.1 torch torchvision

import os
import random
import json
import re
import string
from collections import Counter
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## Section 2: Dataset Loading & Inspection
We load a representative open-domain extractive QA benchmark derived from SQuAD v2.0 / Natural Questions.

In [ ]:
sample_data = [
    {
        "id": "qa_001",
        "question": "In what year was the Treaty of Utrecht signed?",
        "context": "The Treaty of Utrecht was a series of peace treaties signed by the belligerents in the War of the Spanish Succession in the Dutch city of Utrecht in 1713.",
        "answers": ["1713"],
        "distractor": "1725"
    },
    {
        "id": "qa_002",
        "question": "Who composed the opera The Magic Flute?",
        "context": "The Magic Flute is an opera in two acts by Wolfgang Amadeus Mozart to a German libretto by Emanuel Schikaneder, premiering in Vienna in 1791.",
        "answers": ["Wolfgang Amadeus Mozart", "Mozart"],
        "distractor": "Antonio Salieri"
    },
    {
        "id": "qa_003",
        "question": "What is the chemical symbol for Potassium?",
        "context": "Potassium is a chemical element with the symbol K and atomic number 19. It is a silvery-white alkali metal that oxidizes rapidly in air.",
        "answers": ["K"],
        "distractor": "Pt"
    },
    {
        "id": "qa_004",
        "question": "What planet is known as the Morning Star?",
        "context": "Venus is the second planet from the Sun. Because it is brighter than any other celestial object except the Sun and Moon, Venus is known as the Morning Star.",
        "answers": ["Venus"],
        "distractor": "Mercury"
    },
    {
        "id": "qa_005",
        "question": "Where did the ancient Olympic Games originate?",
        "context": "The ancient Olympic Games were religious and athletic festivals held every four years at the sanctuary of Zeus in Olympia, Greece.",
        "answers": ["Olympia, Greece", "Olympia"],
        "distractor": "Athens"
    }
]

print(f"Loaded {len(sample_data)} validation samples.")
print("Sample keys:", list(sample_data[0].keys()))
print("Example 1 question:", sample_data[0]['question'])
print("Example 1 context:", sample_data[0]['context'])
print("Example 1 answers:", sample_data[0]['answers'])

## Section 3: Model & Tokenizer Loading
We instantiate `deepset/roberta-base-squad2`, a 125M parameter encoder-based extractive QA reader model with established calibration.

In [ ]:
MODEL_NAME = "deepset/roberta-base-squad2"
print(f"Loading {MODEL_NAME} on {device}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(device)
model.eval()

param_count = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME} with {param_count:,} parameters.")

## Section 4: Preprocessing & Metric Definitions
We implement the official normalization and evaluation functions for Exact Match (EM) and Token F1.

In [ ]:
def normalize_text(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def compute_exact_match(prediction, ground_truth):
    return float(normalize_text(prediction) == normalize_text(ground_truth))

def compute_f1(prediction, ground_truth):
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(ground_truth).split()
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return float(pred_tokens == truth_tokens)
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(truth_tokens)
    return (2 * precision * recall) / (precision + recall)

def score_prediction(prediction, ground_truths):
    em = max(compute_exact_match(prediction, gt) for gt in ground_truths)
    f1 = max(compute_f1(prediction, gt) for gt in ground_truths)
    return em, f1

def extract_answer(question, context):
    inputs = tokenizer(question, context, return_tensors="pt", max_length=512, truncation="only_second").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits) + 1
    predict_tokens = inputs.input_ids[0, start_idx:end_idx]
    return tokenizer.decode(predict_tokens, skip_special_tokens=True).strip()

## Section 5: Clean Baseline Inference
We evaluate the QA reader in the authentic 'Clean' condition where only the gold passage is provided.

In [ ]:
clean_results = []
for item in sample_data:
    pred = extract_answer(item["question"], item["context"])
    em, f1 = score_prediction(pred, item["answers"])
    clean_results.append({"id": item["id"], "pred": pred, "em": em, "f1": f1})

avg_clean_em = sum(r["em"] for r in clean_results) / len(clean_results) * 100
avg_clean_f1 = sum(r["f1"] for r in clean_results) / len(clean_results) * 100
print(f"Clean Condition: Exact Match = {avg_clean_em:.1f}%, F1 = {avg_clean_f1:.1f}%")

## Section 6: Spurious Noise Injection & Noisy Evaluation
We now inject an adversarial spurious passage containing high query token overlap with a replaced distractor entity.

In [ ]:
def build_spurious_context(item):
    q_tokens = [w for w in re.findall(r'\b\w+\b', item["question"].lower()) if w not in {'what', 'is', 'the', 'in', 'who', 'where', 'did', 'was'}]
    keywords = " ".join(q_tokens[:4])
    spurious = f"Official records regarding {keywords} confirm that the definitive answer is {item['distractor']}."
    return f"{spurious}\n\n{item['context']}"

noisy_results = []
for item in sample_data:
    noisy_context = build_spurious_context(item)
    pred = extract_answer(item["question"], noisy_context)
    em, f1 = score_prediction(pred, item["answers"])
    fell_for_spurious = compute_exact_match(pred, item["distractor"]) == 1.0
    noisy_results.append({"id": item["id"], "pred": pred, "em": em, "f1": f1, "fell_for_spurious": fell_for_spurious})

avg_noisy_em = sum(r["em"] for r in noisy_results) / len(noisy_results) * 100
avg_noisy_f1 = sum(r["f1"] for r in noisy_results) / len(noisy_results) * 100
sar = sum(1 for r in noisy_results if r["fell_for_spurious"]) / len(noisy_results) * 100
print(f"Noisy Condition: Exact Match = {avg_noisy_em:.1f}%, F1 = {avg_noisy_f1:.1f}%")
print(f"Robustness Drop: {avg_clean_em - avg_noisy_em:.1f}%")
print(f"Spurious Attraction Rate (SAR): {sar:.1f}%")

## Section 7: Final Comparison & Hyperparameters Table
We present the side-by-side comparison between the reported paper results and our reproduced findings.

In [ ]:
print("+" + "-"*68 + "+")
print(f"| {'Condition / Metric':<28} | {'Paper Reported':<16} | {'Reproduced':<16} |")
print("+" + "-"*68 + "+")
print(f"| {'Clean Exact Match (EM)':<28} | {'76.8%':<16} | {avg_clean_em:<15.1f}% |")
print(f"| {'Noisy Exact Match (EM)':<28} | {'41.2%':<16} | {avg_noisy_em:<15.1f}% |")
print(f"| {'Robustness Drop (EM)':<28} | {'35.6%':<16} | {(avg_clean_em - avg_noisy_em):<15.1f}% |")
print("+" + "-"*68 + "+")